In [ ]:
# Make this notebook work from either fine-tuning/ or fine-tuning/pre-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


# Lab 14 · Cost Governance at Scale

At 10,000 transcripts a night, the difference between models is the difference between a rounding error and a budget line. This lab turns spend into something you **govern**: model it from real per-token costs, read **live quota/usage**, enforce a **token cap** per request, and wire **budget alerts**. *Predictable spend, no surprises.*


---
## Step 0 — Enable Foundry tracing

*Wire OpenTelemetry to Application Insights so every model call below shows up live in the Microsoft Foundry portal under **your project → Tracing**.*


In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from _tracing import enable_foundry_tracing

enable_foundry_tracing(service_name='acme-live-demo')


---
## Step 1 — Cost model at scale

Project monthly cost across the candidate models using the catalog's relative cost, anchored to a public gpt-4o rate. Same workload, very different bills.


In [ ]:
from _advisor import load_catalog_live

# Anchor: approx blended $/1K tokens for gpt-4o (illustrative).
ANCHOR_PER_1K = 0.0075
TRANSCRIPTS_PER_NIGHT = 10_000
TOKENS_PER_TRANSCRIPT = 1_200
NIGHTS = 30
monthly_tokens = TRANSCRIPTS_PER_NIGHT * TOKENS_PER_TRANSCRIPT * NIGHTS

models, _meta = load_catalog_live()
base_cost = next((m['relative_cost'] for m in models
                  if m['deployment'] == 'gpt-4o' or m.get('model') == 'gpt-4o'), 10)

print(f'{"deployment":28} {"$/1K":>8} {"$/month":>12}')
print('-' * 50)
rows = []
for m in models:
    per_1k = ANCHOR_PER_1K * (m.get('relative_cost', base_cost) / base_cost)
    monthly = per_1k * monthly_tokens / 1000
    rows.append((m['deployment'], per_1k, monthly))
for dep, per_1k, monthly in sorted(rows, key=lambda x: x[2]):
    print(f'{dep:28} {per_1k:>8.4f} {monthly:>12,.0f}')
cheap = min(rows, key=lambda x: x[2]); dear = max(rows, key=lambda x: x[2])
print(f'\nRouting cheapest vs always-premium saves ~${dear[2]-cheap[2]:,.0f}/mo.')


---
## Step 2 — Read live quota / usage (LIVE)

Know your headroom before a launch. Pull current Cognitive Services usage for the resource's region.


In [ ]:
import requests
from _advisor import get_credential

sub  = os.environ['AZURE_SUBSCRIPTION_ID']
rg   = os.environ['AZURE_RESOURCE_GROUP']
acct = os.environ['AZURE_RESOURCE_NAME']
tok  = get_credential().get_token('https://management.azure.com/.default').token
H    = {'Authorization': f'Bearer {tok}'}

# resource region (for the location-scoped usages call)
acc = requests.get(
    f'https://management.azure.com/subscriptions/{sub}/resourceGroups/{rg}'
    f'/providers/Microsoft.CognitiveServices/accounts/{acct}?api-version=2023-05-01',
    headers=H, timeout=15).json()
loc = acc.get('location', 'eastus')

u = requests.get(
    f'https://management.azure.com/subscriptions/{sub}/providers/'
    f'Microsoft.CognitiveServices/locations/{loc}/usages?api-version=2023-05-01',
    headers=H, timeout=15)
items = u.json().get('value', []) if u.status_code == 200 else []
print(f'Quota usage in {loc} (status {u.status_code}):')
for it in items[:8]:
    name = (it.get('name') or {}).get('value', '?')
    print(f"  {name:40} {it.get('currentValue', 0):>8} / {it.get('limit', 0)}")
if not items:
    print('  (none returned - run az login, or quota is unconstrained for this region)')


---
## Step 3 — Enforce a per-request token cap

A runaway prompt shouldn't cost $5. Wrap the client so every call has a hard ceiling and an estimated price you can log.


In [ ]:
from _advisor import try_build_client
client = try_build_client()

MAX_OUTPUT_TOKENS = 256
PER_1K = ANCHOR_PER_1K

def capped_chat(prompt, deployment=None):
    deployment = deployment or os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini')
    if client is None:
        return '[mock]', 0.0
    r = client.chat.completions.create(model=deployment, max_tokens=MAX_OUTPUT_TOKENS,
        messages=[{'role': 'user', 'content': prompt}])
    used = r.usage.total_tokens if r.usage else 0
    return r.choices[0].message.content, round(PER_1K * used / 1000, 5)

ans, cost = capped_chat('Summarize Acme mail-order refill rules in 2 sentences.')
print('Answer:', (ans or '')[:140])
print(f'Capped at {MAX_OUTPUT_TOKENS} output tokens; est. cost this call: ${cost}')


---
## Step 4 — Replace-in-production: budgets + chargeback

Code caps protect a request; **Azure Budgets** protect the bill. Tag deployments per team for chargeback.


In [ ]:
BUDGET_REFERENCE = '''
# --- replace-in-production: subscription budget + alert ---
az consumption budget create --budget-name member-services-ai \\
  --amount 5000 --time-grain Monthly \\
  --category Cost --resource-group $RG \\
  --notifications "Actual_GreaterThan_80=..."  # email/action group at 80%
# Chargeback: tag each deployment, then group Cost Analysis by tag 'team'.
'''
print(BUDGET_REFERENCE)


---
## Takeaways

- Spend is **modeled, not guessed** — Step 1 priced the exact workload.
- **Live quota** tells you headroom before a launch surprises you.
- A **token cap** bounds worst-case cost per request.
- **Budgets + tags** give finance alerts and chargeback.
- Combined with routing (Lab 09), this is where the ROI shows up.

*← The Decision Advisor (Lab 09) routes the `needs_cost_governance` flag here.*
